# 1. RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# 2. [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 2.1 설치
- `pip install ragas rapidfuzz`

## 2.2 RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 2.3 주요 평가지표
### 2.3.1 Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### 2.3.1.1 Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 2.3.1.1.1평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### 2.3.2 Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 2.3.2.1 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## 2.4 Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### 2.4.1 Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 2.4.1.1평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 2.4.1.2 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### 2.4.2 Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 2.4.2.1 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# 3. RAGAS 평가 실습

In [1]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

In [2]:
# docker run -p 6333:6333 -p 6334:6334   -v qdrant_storage:/qdrant/storage   qdrant/qdrant
# termianal에서 실행

In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, SparseVectorParams, VectorParams
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv

load_dotenv()


True

In [4]:
##################################################
# data 준비
##################################################

def load_and_split_olympic_data(file_path="data/olympic_wiki.md"):
    with open(file_path, "r", encoding="utf-8") as fr:
        olympic_text = fr.read()

    # split
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "H1"),
            ("##", "H2"),
            ("###", "H3"),
        ],
    )
    return splitter.split_text(olympic_text)

In [5]:
##################################################
# vector DB 연결
# retriever 생성
##################################################

def get_vectorstore(collection_name: str = "olympic_info_wiki"):
    dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    client = QdrantClient(url="http://localhost:6333")

    # collection 삭제
    if client.collection_exists(collection_name):
        result = client.delete_collection(collection_name=collection_name)

    # collection 생성
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),      
    )
    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,    
        embedding=dense_embeddings
    )
    
    # document 추가
    documents = load_and_split_olympic_data()
    vectorstore.add_documents(documents=documents)
    
    return vectorstore

def get_retriever(vectorstore, k: int = 5):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever

In [6]:
vectorstore = get_vectorstore()
print(vectorstore)
retriever = get_retriever(vectorstore)
print(retriever)

tags=['QdrantVectorStore', 'OpenAIEmbeddings'] vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x00000181384B2660> search_kwargs={'k': 5}


In [7]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from operator import itemgetter

vectorstore = get_vectorstore()
retriever = get_retriever(vectorstore)

prompt_txt = """<instruction>
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
"""
prompt = ChatPromptTemplate.from_template(
    template=prompt_txt
)

model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

def format_doc_to_str(documents:list[Document])->list[str]:
    """
    VectorStore에 조회한 문서들(list[Document])에서 내용(page_content)만 추출해서 list[str] 로 반환.
    RAGAS 평가시 context는 각 검색한 문서를 list[str] 로 받기 때문에 이렇게 처리.
    
    Args:
        documents(list[Document]): [Document(..), Document(...), ..]}
    Returns:
        list[str]: 각 문서의 내용만 추출해서 리스트에 담는다.
    """
    return [doc.page_content for doc in documents]

# RAG 체인 -> RAG 평가 데이터셋을 만드는 RAG Chain -> 최종응답: LLM의 응답(str), 검색한 문서들(list[str])
chain = RunnablePassthrough() | {
    "context":retriever | format_doc_to_str,
    "query":RunnablePassthrough()
} | { 
    "response": prompt | model | parser, # LLM 응답
    "retrieved_context": itemgetter("context") # 검색한 문서들 반환
}
# RunnablePassthrough() -> LCEL 체인을 만들려면 구성요소중 하나가 Runnable이어야 하는데 이 체인은 dict |dict 구조이기 때문에 추가헀다. 
## 추가하지 않으면  | 연산(or 연산이 된다)

In [8]:
res = chain.invoke("1회 올림픽은 언제 어디서 열렸지")

In [9]:
print(res.keys())
print(res)

dict_keys(['response', 'retrieved_context'])
{'response': '정보가 부족해 답을 할 수없습니다.', 'retrieved_context': ['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에

In [10]:
res["response"]

'정보가 부족해 답을 할 수없습니다.'

In [11]:
res["retrieved_context"]

['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스 

In [12]:
res

{'response': '정보가 부족해 답을 할 수없습니다.',
 'retrieved_context': ['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요

# 4. RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## 4.1 TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.


> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 > 넣는다.
>    ```python
>       try:
>           from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [13]:
# !uv pip install langchain_google_vertexai

In [14]:
# 주피터노트북 환경에서 비동기적 처리 위해 
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [15]:
from ragas.testset import TestsetGenerator

In [16]:
#
# testset -> Context들(문서들) - [질문 - 정답답변 + (Retriever가 찾은 문서 + LLM 응답: chain에서 생성)]
# 1. Context(문서들)을 추출 - TestsetGenerator -> 질문과 정답 답변 생성

import random

# 데이터셋을 생성할 때 사용할 Context를 추출
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "olympic_info_wiki"

info = client.get_collection(COLLECTION_NAME)
total_docs = info.points_count

results, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=total_docs,
)

# random하게 k(5)개를 sampling
sample_docs:"list[PointStruct]" =random.sample(results, 5) # 리스트에서 랜덤하게 k(5)개를 추출

# PointStruct - payload: page_content, metadata
# page_content만 추출해서 list[str]
docs = [point.payload['page_content'] for point in sample_docs]



In [44]:
total_docs

25

In [43]:
sample_docs

[Record(id='88e526a7-42b7-4295-9066-febd4170a039', payload={'page_content': '국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식적으로 메달집계에 관해 수많은 기록들이 남아있다. 메달집계는 현대 올림픽에서 성공한 올림픽 선수들을 알아보는 방법이기도 하다. 아래의 표는 올림픽에서 개인 통산 다관왕을 한 상위 10명의 선수들에 관한 표이다.  \n| 선수 | 성별 | 나라 | 종목 | 올림픽 참가 기간 | 금 | 은 | 동 | 합계 |\n|------|------|------|------|------------------|----|----|----|----|\n| 마이클 펠프스 | 남성 | 미국 | 수영 | 2000년~2016년 | 26 | 3 | 2 | 31 |\n| 라리사 라티니나 | 여성 | 소련 | 체조 | 1956년~1964년 | 9 | 5 | 4 | 18 |\n| 파보 누르미 | 남성 | 핀란드 | 육상 | 1920년~1928년 | 9 | 3 | 0 | 12 |\n| 마크 스피츠 | 남성 | 미국 | 수영 | 1968년~1972년 | 9 | 1 | 1 | 11 |\n| 칼 루이스 | 남성 | 미국 | 육상 | 1984년~1996년 | 9 | 1 | 0 | 10 |\n| 올레 에이나르 비에른달렌 | 남성 | 노르웨이 | 크로스 컨트리 | 1998~2014년 | 8 | 4 | 1 | 13 |\n| 비르기트 프린츠 | 여성 | 동독/독일 | 카누 | 1980년~2004년 | 8 | 4 | 0 | 12 |\n| 가토 사와오 | 남성 | 일본 | 체조 | 1968년~1976년 | 8 | 3 | 1 | 12 |\n| 제니 톰슨 | 여성 | 미국 | 수영 | 1992년~2004년 | 8 | 3 | 1 | 12 |\n| 맷 비온디 | 남성 | 미국 | 수영 | 1984년~1992년 | 8 | 2 | 1 | 11 |', 'metadata': {'H1': '올림픽', 'H2': '우

In [18]:
docs

['국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식적으로 메달집계에 관해 수많은 기록들이 남아있다. 메달집계는 현대 올림픽에서 성공한 올림픽 선수들을 알아보는 방법이기도 하다. 아래의 표는 올림픽에서 개인 통산 다관왕을 한 상위 10명의 선수들에 관한 표이다.  \n| 선수 | 성별 | 나라 | 종목 | 올림픽 참가 기간 | 금 | 은 | 동 | 합계 |\n|------|------|------|------|------------------|----|----|----|----|\n| 마이클 펠프스 | 남성 | 미국 | 수영 | 2000년~2016년 | 26 | 3 | 2 | 31 |\n| 라리사 라티니나 | 여성 | 소련 | 체조 | 1956년~1964년 | 9 | 5 | 4 | 18 |\n| 파보 누르미 | 남성 | 핀란드 | 육상 | 1920년~1928년 | 9 | 3 | 0 | 12 |\n| 마크 스피츠 | 남성 | 미국 | 수영 | 1968년~1972년 | 9 | 1 | 1 | 11 |\n| 칼 루이스 | 남성 | 미국 | 육상 | 1984년~1996년 | 9 | 1 | 0 | 10 |\n| 올레 에이나르 비에른달렌 | 남성 | 노르웨이 | 크로스 컨트리 | 1998~2014년 | 8 | 4 | 1 | 13 |\n| 비르기트 프린츠 | 여성 | 동독/독일 | 카누 | 1980년~2004년 | 8 | 4 | 0 | 12 |\n| 가토 사와오 | 남성 | 일본 | 체조 | 1968년~1976년 | 8 | 3 | 1 | 12 |\n| 제니 톰슨 | 여성 | 미국 | 수영 | 1992년~2004년 | 8 | 3 | 1 | 12 |\n| 맷 비온디 | 남성 | 미국 | 수영 | 1984년~1992년 | 8 | 2 | 1 | 11 |',
 '올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개최지 선정에는 약 2년이 걸린다. 유치를 희망하는 도시는 우선 자국의 올림픽 위원회에 신청을 해야 한다. 만약 

In [ ]:
# testset 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
#testsetgenerator 는 gpt-5 이후 버전은 사용할 수 없다. 
## Langchain의 모델과 Embedding 모델 -> ragas에서 사용할 수 있도록 변환(wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="""
- 사람들이 올림픽에 대해서 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법읋 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
""" 
# 질문/답변을 생성할 때 LLM에게 전달할 system Prompt를 설정
)





C:\Users\Playdata\AppData\Local\Temp\ipykernel_36440\833878933.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_36440\833878933.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [20]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Context 내용 테스트셋 개수(질문 답변 개수)
)

Applying SummaryExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

Node 3ffc6af0-a7b0-424d-87d7-9903e30716e0 does not have a summary. Skipping filtering.
Node 91653b0d-cbed-44ca-8134-dfc62fff9379 does not have a summary. Skipping filtering.
Node e8dd9f9b-0aa5-406d-b235-7b2472e6f0df does not have a summary. Skipping filtering.
Node e0768294-1d60-413c-a40a-a6bdb692e01f does not have a summary. Skipping filtering.
Node 1147b0dc-49c8-49db-b8fe-f5011e3a8fc5 does not have a summary. Skipping filtering.


Applying EmbeddingExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [21]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='칼 루이스는 올림픽에서 몇 개의 금메달을 획득했나요?', retrieved_contexts=None, reference_contexts=['국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식적으로 메달집계에 관해 수많은 기록들이 남아있다. 메달집계는 현대 올림픽에서 성공한 올림픽 선수들을 알아보는 방법이기도 하다. 아래의 표는 올림픽에서 개인 통산 다관왕을 한 상위 10명의 선수들에 관한 표이다.  \n| 선수 | 성별 | 나라 | 종목 | 올림픽 참가 기간 | 금 | 은 | 동 | 합계 |\n|------|------|------|------|------------------|----|----|----|----|\n| 마이클 펠프스 | 남성 | 미국 | 수영 | 2000년~2016년 | 26 | 3 | 2 | 31 |\n| 라리사 라티니나 | 여성 | 소련 | 체조 | 1956년~1964년 | 9 | 5 | 4 | 18 |\n| 파보 누르미 | 남성 | 핀란드 | 육상 | 1920년~1928년 | 9 | 3 | 0 | 12 |\n| 마크 스피츠 | 남성 | 미국 | 수영 | 1968년~1972년 | 9 | 1 | 1 | 11 |\n| 칼 루이스 | 남성 | 미국 | 육상 | 1984년~1996년 | 9 | 1 | 0 | 10 |\n| 올레 에이나르 비에른달렌 | 남성 | 노르웨이 | 크로스 컨트리 | 1998~2014년 | 8 | 4 | 1 | 13 |\n| 비르기트 프린츠 | 여성 | 동독/독일 | 카누 | 1980년~2004년 | 8 | 4 | 0 | 12 |\n| 가토 사와오 | 남성 | 일본 | 체조 | 1968년~1976년 | 8 | 3 | 1 | 12 |\n| 제니 톰슨 | 여성 | 미국 | 수영 | 1992년~2004년 | 8 | 3 | 1 | 12 |\n| 맷 비온디 | 남성 | 미국 |

In [53]:
sample1 = testset.samples[0].eval_sample
print("사용자질문:", sample1.user_input)
print("Context:", sample1.reference_contexts)
print("생성된 답변(정답):", sample1.reference)
print("평가대상 RAG의 답변:", sample1.response)
print("평가대상 RAG가 검색한 Context:", sample1.retrieved_contexts)


사용자질문: 칼 루이스는 올림픽에서 몇 개의 금메달을 획득했나요?
Context: ['국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식적으로 메달집계에 관해 수많은 기록들이 남아있다. 메달집계는 현대 올림픽에서 성공한 올림픽 선수들을 알아보는 방법이기도 하다. 아래의 표는 올림픽에서 개인 통산 다관왕을 한 상위 10명의 선수들에 관한 표이다.  \n| 선수 | 성별 | 나라 | 종목 | 올림픽 참가 기간 | 금 | 은 | 동 | 합계 |\n|------|------|------|------|------------------|----|----|----|----|\n| 마이클 펠프스 | 남성 | 미국 | 수영 | 2000년~2016년 | 26 | 3 | 2 | 31 |\n| 라리사 라티니나 | 여성 | 소련 | 체조 | 1956년~1964년 | 9 | 5 | 4 | 18 |\n| 파보 누르미 | 남성 | 핀란드 | 육상 | 1920년~1928년 | 9 | 3 | 0 | 12 |\n| 마크 스피츠 | 남성 | 미국 | 수영 | 1968년~1972년 | 9 | 1 | 1 | 11 |\n| 칼 루이스 | 남성 | 미국 | 육상 | 1984년~1996년 | 9 | 1 | 0 | 10 |\n| 올레 에이나르 비에른달렌 | 남성 | 노르웨이 | 크로스 컨트리 | 1998~2014년 | 8 | 4 | 1 | 13 |\n| 비르기트 프린츠 | 여성 | 동독/독일 | 카누 | 1980년~2004년 | 8 | 4 | 0 | 12 |\n| 가토 사와오 | 남성 | 일본 | 체조 | 1968년~1976년 | 8 | 3 | 1 | 12 |\n| 제니 톰슨 | 여성 | 미국 | 수영 | 1992년~2004년 | 8 | 3 | 1 | 12 |\n| 맷 비온디 | 남성 | 미국 | 수영 | 1984년~1992년 | 8 | 2 | 1 | 11 |']
생성된 답변(정답): 칼 루이스는 올림픽에서 금메달 9개를 획득했습니다.
평가대상 RAG의 답변: None
평가대

In [54]:
# 생성된 Testset을 Pandas DataFrame으로 변환
eval_df = testset.to_pandas()

In [24]:
eval_df.shape

(10, 7)

In [56]:
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,칼 루이스는 올림픽에서 몇 개의 금메달을 획득했나요?,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식...",칼 루이스는 올림픽에서 금메달 9개를 획득했습니다.,Olympic Sports Historian,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,비르기트 프린츠 올림픽 메달 몇개 땄나요?,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식...","비르기트 프린츠는 올림픽에서 금메달 8개, 은메달 4개, 동메달 0개로 총 12개의...",Olympic Sports Historian,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
2,중국은 올림픽에서 어떤 기록을 가지고 있나요?,[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...,2008년 하계 올림픽 때 가장 많은 선수가 참여한 나라는 중국으로 639명이 참가...,Youth Sports Coordinator,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
3,미국은 올림픽을 몇 번 개최했습니까?,[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...,미국은 5번의 하계 올림픽과 4번의 동계 올림픽을 개최하면서 최다 올림픽을 개최한 ...,Olympic Sports Historian,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
4,2010년 뭐 있었어?,[2010년에 첫대회가 개최되며 14~18세가 참가하는 대회이다. 청소년 올림픽은 ...,2010년에 첫대회가 개최되며 14~18세가 참가하는 대회이다.,Youth Sports Coordinator,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
5,"2001년 무슨 일 있었는지, 청소년 올림픽이랑 관련 있는지 설명해 주세요, 그리고...",[2010년에 첫대회가 개최되며 14~18세가 참가하는 대회이다. 청소년 올림픽은 ...,"청소년 올림픽은 IOC위원장인 자크 로게가 2001년에 고안한 것으로써, 2001년...",Olympic Sports Historian,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
6,제2차 세계대전이 올림픽에 어떤 영향을 미쳤나요?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",제2차 세계대전으로 인해 일본 도쿄에서 열리기로 했던 제12회 1940년 하계 올림...,Youth Sports Coordinator,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
7,쿠베르탱 올림픽 평화 맞아?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...","쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다.",Olympic Bid Coordinator,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
8,솔트레이크 시티 올림픽 유치할때 IOC 위원들이 뇌물받았다는 얘기 있는데 그거 어떻...,[국제 올림픽 위원회(이하 IOC로 지칭)는 몇몇 위원들이 한 행위에 대해서 비판을...,1998년에 몇몇 IOC위원들이 2002년 솔트레이크 시티 동계 올림픽 유치 과정에...,Olympic Bid Coordinator,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
9,"에이버리 브런디지 누구고 IOC에서 뭐했는지, 그리고 왜 비난받았는지 자세히 설명해줘요.",[국제 올림픽 위원회(이하 IOC로 지칭)는 몇몇 위원들이 한 행위에 대해서 비판을...,에이버리 브런디지는 국제 올림픽 위원회(IOC) 위원장이었고 20년 넘게 위원장직을...,Youth Sports Coordinator,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer


In [26]:
row_idx = 0
q = eval_df.loc[row_idx, 'user_input']
resp = chain.invoke(q)

In [27]:
resp['response']

'칼 루이스는 올림픽에서 **금메달 9개**를 획득했습니다.'

In [28]:
resp['retrieved_context']

['국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식적으로 메달집계에 관해 수많은 기록들이 남아있다. 메달집계는 현대 올림픽에서 성공한 올림픽 선수들을 알아보는 방법이기도 하다. 아래의 표는 올림픽에서 개인 통산 다관왕을 한 상위 10명의 선수들에 관한 표이다.  \n| 선수 | 성별 | 나라 | 종목 | 올림픽 참가 기간 | 금 | 은 | 동 | 합계 |\n|------|------|------|------|------------------|----|----|----|----|\n| 마이클 펠프스 | 남성 | 미국 | 수영 | 2000년~2016년 | 26 | 3 | 2 | 31 |\n| 라리사 라티니나 | 여성 | 소련 | 체조 | 1956년~1964년 | 9 | 5 | 4 | 18 |\n| 파보 누르미 | 남성 | 핀란드 | 육상 | 1920년~1928년 | 9 | 3 | 0 | 12 |\n| 마크 스피츠 | 남성 | 미국 | 수영 | 1968년~1972년 | 9 | 1 | 1 | 11 |\n| 칼 루이스 | 남성 | 미국 | 육상 | 1984년~1996년 | 9 | 1 | 0 | 10 |\n| 올레 에이나르 비에른달렌 | 남성 | 노르웨이 | 크로스 컨트리 | 1998~2014년 | 8 | 4 | 1 | 13 |\n| 비르기트 프린츠 | 여성 | 동독/독일 | 카누 | 1980년~2004년 | 8 | 4 | 0 | 12 |\n| 가토 사와오 | 남성 | 일본 | 체조 | 1968년~1976년 | 8 | 3 | 1 | 12 |\n| 제니 톰슨 | 여성 | 미국 | 수영 | 1992년~2004년 | 8 | 3 | 1 | 12 |\n| 맷 비온디 | 남성 | 미국 | 수영 | 1984년~1992년 | 8 | 2 | 1 | 11 |',
 '개인 혹은 팀으로 경기에 출전해서 1위, 2위, 3위를 한 선수는 메달을 받는다. 1912년까지는 우승자에게 순금으로 된 금메달을 주었으며 그 후에는 도금된 금메달을 준다. 하지만, 2010 

In [29]:
# 모든 testset의 데이터데 대한 llm 응답과 retriever의 검색 겷과를 추가
response_list = []
retrieved_context_list = []

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp['response'])
    retrieved_context_list.append(resp['retrieved_context'])

In [30]:
print(len(response_list), len(retrieved_context_list))

10 10


In [31]:
response_list

['칼 루이스는 올림픽에서 9개의 금메달을 획득했습니다.',
 '비르기트 프린츠는 올림픽에서 총 12개의 메달을 땄습니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '미국은 5번의 하계 올림픽과 4번의 동계 올림픽을 개최하여, 총 9번 개최했습니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '2001년에는 **IOC 위원장인 자크 로게가 청소년 올림픽을 고안**했습니다.  \n즉, **청소년 올림픽과 직접 관련이 있는 해**입니다.\n\n정리하면:\n- **누가:** 자크 로게\n- **무엇을:** 청소년 올림픽을 **고안**함\n- **언제:** 2001년\n\n또한 이 청소년 올림픽은 이후 **2007년 제119차 IOC 총회에서 승인**되었습니다.',
 '제2차 세계대전 때는 일본 도쿄에서 열리기로 했던 제12회 1940년 하계 올림픽, 삿포로에서 열리기로 했던 1940년 동계 올림픽, 영국 런던에서 열리기로 했던 제13회 1944년 하계 올림픽, 이탈리아 코르티나담페초에서 열릴 예정인 1944년 동계 올림픽이 취소되었습니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '솔트레이크 시티 2002년 동계 올림픽 유치 과정에서는, 몇몇 IOC 위원들이 미국 측으로부터 “솔트레이크 시티를 개최지로 뽑아달라”는 뇌물 청탁을 받았다는 사실이 1998년에 폭로되었습니다.\n\n그 뒤 IOC는 다음과 같은 조사를 했습니다.\n- 사퇴한 IOC 위원 4명\n- 강제 퇴출된 6명\n\n이 스캔들은 이후 유치 과정에서 같은 문제가 다시 일어나지 않도록 하는 계기가 되어, IOC가 개혁에 착수하게 만든 긍정적인 역할도 했습니다.\n\n즉,  \n1) 뇌물 청탁 폭로가 있었고  \n2) IOC가 관련 위원들을 조사·처분했으며  \n3) 그 사건 이후 개최지 선정 과정의 개혁이 이루어졌습니다.',
 '에이버리 브런디지는 IOC 위원장이었던 인물입니다.  \n제공된 context에 따르면 그는 **1952년부터 1972년까지 IOC 위원장직을 맡았고**, 이 

In [32]:
retrieved_context_list

[['국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식적으로 메달집계에 관해 수많은 기록들이 남아있다. 메달집계는 현대 올림픽에서 성공한 올림픽 선수들을 알아보는 방법이기도 하다. 아래의 표는 올림픽에서 개인 통산 다관왕을 한 상위 10명의 선수들에 관한 표이다.  \n| 선수 | 성별 | 나라 | 종목 | 올림픽 참가 기간 | 금 | 은 | 동 | 합계 |\n|------|------|------|------|------------------|----|----|----|----|\n| 마이클 펠프스 | 남성 | 미국 | 수영 | 2000년~2016년 | 26 | 3 | 2 | 31 |\n| 라리사 라티니나 | 여성 | 소련 | 체조 | 1956년~1964년 | 9 | 5 | 4 | 18 |\n| 파보 누르미 | 남성 | 핀란드 | 육상 | 1920년~1928년 | 9 | 3 | 0 | 12 |\n| 마크 스피츠 | 남성 | 미국 | 수영 | 1968년~1972년 | 9 | 1 | 1 | 11 |\n| 칼 루이스 | 남성 | 미국 | 육상 | 1984년~1996년 | 9 | 1 | 0 | 10 |\n| 올레 에이나르 비에른달렌 | 남성 | 노르웨이 | 크로스 컨트리 | 1998~2014년 | 8 | 4 | 1 | 13 |\n| 비르기트 프린츠 | 여성 | 동독/독일 | 카누 | 1980년~2004년 | 8 | 4 | 0 | 12 |\n| 가토 사와오 | 남성 | 일본 | 체조 | 1968년~1976년 | 8 | 3 | 1 | 12 |\n| 제니 톰슨 | 여성 | 미국 | 수영 | 1992년~2004년 | 8 | 3 | 1 | 12 |\n| 맷 비온디 | 남성 | 미국 | 수영 | 1984년~1992년 | 8 | 2 | 1 | 11 |',
  '개인 혹은 팀으로 경기에 출전해서 1위, 2위, 3위를 한 선수는 메달을 받는다. 1912년까지는 우승자에게 순금으로 된 금메달을 주었으며 그 후에는 도금된 금메달을 준다. 하지만, 201

In [33]:
#
# eval_df에 컬럼으로 추가
#
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,response,retrieved_contexts
0,칼 루이스는 올림픽에서 몇 개의 금메달을 획득했나요?,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식...",칼 루이스는 올림픽에서 금메달 9개를 획득했습니다.,Olympic Sports Historian,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,칼 루이스는 올림픽에서 9개의 금메달을 획득했습니다.,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식..."
1,비르기트 프린츠 올림픽 메달 몇개 땄나요?,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식...","비르기트 프린츠는 올림픽에서 금메달 8개, 은메달 4개, 동메달 0개로 총 12개의...",Olympic Sports Historian,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer,비르기트 프린츠는 올림픽에서 총 12개의 메달을 땄습니다.,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식..."
2,중국은 올림픽에서 어떤 기록을 가지고 있나요?,[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...,2008년 하계 올림픽 때 가장 많은 선수가 참여한 나라는 중국으로 639명이 참가...,Youth Sports Coordinator,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식..."
3,미국은 올림픽을 몇 번 개최했습니까?,[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...,미국은 5번의 하계 올림픽과 4번의 동계 올림픽을 개최하면서 최다 올림픽을 개최한 ...,Olympic Sports Historian,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,"미국은 5번의 하계 올림픽과 4번의 동계 올림픽을 개최하여, 총 9번 개최했습니다.",[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...
4,2010년 뭐 있었어?,[2010년에 첫대회가 개최되며 14~18세가 참가하는 대회이다. 청소년 올림픽은 ...,2010년에 첫대회가 개최되며 14~18세가 참가하는 대회이다.,Youth Sports Coordinator,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[올림픽에서 첫 번째 보이콧은 1956년 하계 올림픽에서 시작되었다. 네덜란드, 스..."
5,"2001년 무슨 일 있었는지, 청소년 올림픽이랑 관련 있는지 설명해 주세요, 그리고...",[2010년에 첫대회가 개최되며 14~18세가 참가하는 대회이다. 청소년 올림픽은 ...,"청소년 올림픽은 IOC위원장인 자크 로게가 2001년에 고안한 것으로써, 2001년...",Olympic Sports Historian,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,2001년에는 **IOC 위원장인 자크 로게가 청소년 올림픽을 고안**했습니다. ...,[2010년에 첫대회가 개최되며 14~18세가 참가하는 대회이다. 청소년 올림픽은 ...
6,제2차 세계대전이 올림픽에 어떤 영향을 미쳤나요?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",제2차 세계대전으로 인해 일본 도쿄에서 열리기로 했던 제12회 1940년 하계 올림...,Youth Sports Coordinator,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,"제2차 세계대전 때는 일본 도쿄에서 열리기로 했던 제12회 1940년 하계 올림픽,...","[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실..."
7,쿠베르탱 올림픽 평화 맞아?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...","쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다.",Olympic Bid Coordinator,MISSPELLED,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실..."
8,솔트레이크 시티 올림픽 유치할때 IOC 위원들이 뇌물받았다는 얘기 있는데 그거 어떻...,[국제 올림픽 위원회(이하 IOC로 지칭)는 몇몇 위원들이 한 행위에 대해서 비판을...,1998년에 몇몇 IOC위원들이 2002년 솔트레이크 시티 동계 올림픽 유치 과정에...,Olympic Bid Coordinator,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,"솔트레이크 시티 2002년 동계 올림픽 유치 과정에서는, 몇몇 IOC 위원들이 미국...",[국제 올림픽 위원회(이하 IOC로 지칭)는 몇몇 위원들이 한 행위에 대해서 비판을...
9,"에이버리 브런디지 누구고 IOC에서 뭐했는지, 그리고 왜 비난받았는지 자세히 설명해줘요.",[국제 올림픽 위원회(이하 IOC로 지칭)는 몇몇 위원들이 한 행위에 대해서 비판을...,에이버리 브런디지는 국제 올림픽 위원회(IOC) 위원장이었고 20년 넘게 위원장직을...,Youth Sports Coordinator,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,에이버리 브런디지는 IOC 위원장이었던 인물입니다. \n제공된 context에 따...,[국제 올림픽 위원회(이하 IOC로 지칭)는 몇몇 위원들이 한 행위에 대해서 비판을...


In [34]:
# eval_df를 ragas의 평가데이터셋 타입으로 변환
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=10)

In [35]:
#
# 평가
#
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    Faithfulness,
    AnswerRelevancy,
)
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_36440\3378832119.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_36440\3378832119.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_36440\3378832119.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\User

In [37]:
# 평가할 때 사용할 LLM embedding 모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)

]
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_36440\384477035.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_36440\384477035.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Exception raised in Job[2]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[16]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[22]: TimeoutError()
Exception raised in Job[24]: TimeoutError()


In [38]:
eval_result

{'context_recall': 1.0000, 'llm_context_precision_with_reference': 0.8056, 'faithfulness': 0.7222, 'answer_relevancy': 0.4963}

In [39]:
result_df = eval_result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,칼 루이스는 올림픽에서 몇 개의 금메달을 획득했나요?,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식...",칼 루이스는 올림픽에서 9개의 금메달을 획득했습니다.,칼 루이스는 올림픽에서 금메달 9개를 획득했습니다.,1.0,1.000000,NaN,0.989767
1,비르기트 프린츠 올림픽 메달 몇개 땄나요?,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식...",비르기트 프린츠는 올림픽에서 총 12개의 메달을 땄습니다.,"비르기트 프린츠는 올림픽에서 금메달 8개, 은메달 4개, 동메달 0개로 총 12개의...",1.0,1.000000,NaN,0.900890
2,중국은 올림픽에서 어떤 기록을 가지고 있나요?,"[국제 올림픽 위원회(IOC)에서는 공식적으로 메달집계를 하고 있지 않지만, 비공식...",정보가 부족해 답을 할 수없습니다.,2008년 하계 올림픽 때 가장 많은 선수가 참여한 나라는 중국으로 639명이 참가...,1.0,0.500000,NaN,0.000000
3,미국은 올림픽을 몇 번 개최했습니까?,[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...,"미국은 5번의 하계 올림픽과 4번의 동계 올림픽을 개최하여, 총 9번 개최했습니다.",미국은 5번의 하계 올림픽과 4번의 동계 올림픽을 개최하면서 최다 올림픽을 개최한 ...,1.0,1.000000,NaN,0.962999
4,2010년 뭐 있었어?,"[올림픽에서 첫 번째 보이콧은 1956년 하계 올림픽에서 시작되었다. 네덜란드, 스...",정보가 부족해 답을 할 수없습니다.,2010년에 첫대회가 개최되며 14~18세가 참가하는 대회이다.,NaN,0.000000,NaN,0.000000
5,"2001년 무슨 일 있었는지, 청소년 올림픽이랑 관련 있는지 설명해 주세요, 그리고...",[2010년에 첫대회가 개최되며 14~18세가 참가하는 대회이다. 청소년 올림픽은 ...,2001년에는 **IOC 위원장인 자크 로게가 청소년 올림픽을 고안**했습니다. ...,"청소년 올림픽은 IOC위원장인 자크 로게가 2001년에 고안한 것으로써, 2001년...",NaN,1.000000,NaN,0.488485
6,제2차 세계대전이 올림픽에 어떤 영향을 미쳤나요?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...","제2차 세계대전 때는 일본 도쿄에서 열리기로 했던 제12회 1940년 하계 올림픽,...",제2차 세계대전으로 인해 일본 도쿄에서 열리기로 했던 제12회 1940년 하계 올림...,NaN,1.000000,1.000000,0.430096
7,쿠베르탱 올림픽 평화 맞아?,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",정보가 부족해 답을 할 수없습니다.,"쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다.",1.0,0.805556,0.000000,0.000000
8,솔트레이크 시티 올림픽 유치할때 IOC 위원들이 뇌물받았다는 얘기 있는데 그거 어떻...,[국제 올림픽 위원회(이하 IOC로 지칭)는 몇몇 위원들이 한 행위에 대해서 비판을...,"솔트레이크 시티 2002년 동계 올림픽 유치 과정에서는, 몇몇 IOC 위원들이 미국...",1998년에 몇몇 IOC위원들이 2002년 솔트레이크 시티 동계 올림픽 유치 과정에...,1.0,1.000000,0.888889,0.604036
9,"에이버리 브런디지 누구고 IOC에서 뭐했는지, 그리고 왜 비난받았는지 자세히 설명해줘요.",[국제 올림픽 위원회(이하 IOC로 지칭)는 몇몇 위원들이 한 행위에 대해서 비판을...,에이버리 브런디지는 IOC 위원장이었던 인물입니다. \n제공된 context에 따...,에이버리 브런디지는 국제 올림픽 위원회(IOC) 위원장이었고 20년 넘게 위원장직을...,1.0,0.750000,1.000000,0.586448
